# Paper plots — KL-diag protagonist (Polar-LoRA)

Locked config: `kl-diag-polar-lora` · PolarExpress PE=8 · Nesterov · β₁=0.9 · δ=1e-4 · k=1 · curvature_beta=0.99 · precond_refresh_every=10 · **precond_method=gram_ns**. AdamW + iMuon baselines; σ=0.0017 noise floor.

In-flight runs render partially (`allow_partial=True`).

In [ ]:
%load_ext autoreload
%autoreload 2

# Every definition -- arm predicates, the per-cell run cache, the panel functions --
# lives in lora_playground/plotting/paper_plots_lib.py, NOT in these cells. Reach
# through P (P.PROTO, P.rr_slot_panel, ...): with autoreload on, an edit to that file
# is picked up on the next cell you run, with no "which cell defines this again?".
# Definitions used to sit in cells 1/33/35/47, which is why a fix on disk could sit
# there while the notebook kept raising the error it fixed.
import lora_playground.plotting.paper_plots_lib as P

# Legacy aliases for Figures 17-20, whose cells still carry their own plotting code and
# have not been moved into the module yet. Names bound HERE do not track edits to the
# module -- that is the staleness this restructure removes -- so use P.<name> in any new
# cell, and treat this block as temporary.
from lora_playground.plotting.paper_plots_lib import (
    ROOT, SIGMA, ADAMW, PROTO, cell_runs,
    om as _om, pred_matches as _pred_matches, variant_key_fn as _variant_key_fn,
)
from lora_playground.plotting import compare_variants_figure


## Per-cell panels — AdamW vs Polar-LoRA (kl-diag) vs iMuon vs naive factor-Muon vs the double-ablation (w/o curvature+magnitude); baseline arms render in any panel where their runs exist (in-flight runs show from first eval)

### Figure 1. OLMo-2-1B opc r256

In [ ]:
P.panel_n(0)   # OLMo-2-1B opc r256


### Figure 2. Qwen2.5-1.5B opc r256

In [ ]:
P.panel_n(1)   # Qwen2.5-1.5B opc r256


### Figure 3. Llama-3.2-1B opc r256

In [ ]:
P.panel_n(2)   # Llama-3.2-1B opc r256


### Figure 4. Llama-3-8B opc r256

In [ ]:
P.panel_n(3)   # Llama-3-8B opc r256


### Figure 5. Qwen2.5-1.5B bengali r256  (OOD pair vs cell 2)

In [ ]:
P.panel_n(4)   # Qwen2.5-1.5B bengali r256


## Math model breadth — openmath r256

Parallel to the opc breadth panels above: only the base model varies, rank fixed at $r=256$, finetuning corpus is OpenMathInstruct-2. Llama-3.2-1B openmath r256 is the rank-ladder top cell below. Each panel: AdamW vs Polar-LoRA (kl-diag); iMuon/Muon arms render only where their runs exist (`allow_partial`).

### Figure 6. OLMo-2-1B openmath r256  (math breadth)

In [ ]:
P.panel_n(5)   # OLMo-2-1B openmath r256


### Figure 7. Qwen2.5-1.5B openmath r256  (math breadth)

In [ ]:
P.panel_n(6)   # Qwen2.5-1.5B openmath r256


### Figure 8. Llama-3-8B openmath r256  (math breadth)

In [ ]:
P.panel_n(7)   # Llama-3-8B openmath r256


### Figure 9. Llama-3.2-1B openmath r16  (rank ladder — low rank, common LoRA regime)

In [ ]:
P.panel_n(8)   # Llama-3.2-1B openmath r16


### Figure 10. Llama-3.2-1B openmath r32  (rank ladder — low rank)

In [ ]:
P.panel_n(9)   # Llama-3.2-1B openmath r32


### Figure 11. Llama-3.2-1B openmath r64  (rank ladder)

In [ ]:
P.panel_n(10)   # Llama-3.2-1B openmath r64


### Figure 12. Llama-3.2-1B openmath r128  (rank ladder)

In [ ]:
P.panel_n(11)   # Llama-3.2-1B openmath r128


### Figure 13. Llama-3.2-1B openmath r256  (rank ladder)

In [ ]:
P.panel_n(12)   # Llama-3.2-1B openmath r256


## Figure 14. E2 ablation — Llama-3.2-1B openmath, per rank (leave-one-out)

One panel per rank. Arms: **Polar-LoRA (kl-diag)** vs **w/o curvature control** (drop the
curvature EMAs → identity metric; the novelty over the iMuon family) vs
**w/o magnitude control** (`cw_unpinned`: remove the operator-norm magnitude rule → true-scale,
no pin) vs **w/o curvature+magnitude (LoRA-Muon step)**. Δ vs Polar-LoRA (kl-diag) in σ-units,
step-matched (`allow_partial`). The w/o-curvature arm runs at r256 only (the canonical rank).
Polar is *not* ablated (cited to the spectral-method literature).

In [ ]:
P.ablation_panel(256)


## Figure 15. Derivation ablations — Llama-3.2-1B openmath r256

Two arms, each removing exactly one premise of the PoLoRA derivation
(main.tex §2.3 "Per-Sample Loss Control" and the "Direction" paragraph), with
every other knob held at the protagonist config (β₁=0.9, δ=1e-4, `gram_ns`
inverse-sqrt, PolarExpress-8, Nesterov, k=1, 9000 steps).

| arm | `--optimizer` | left/right metric for $\Delta A$ | direction operator |
|---|---|---|---|
| Polar-LoRA (kl-diag) | `kl-diag-polar-lora` | $C_B=B^\top P B$, $Q$ | $\mathrm{msign}$ |
| w/o per-sample loss control (averaged) | `kl-diag-lora` | $C_B=B^\top P B$, $Q$ | identity |
| w/o product structure (factorwise) | `kl-shampoo-polar-lora` | $P_A$ (own Gram), $Q_A$ | $\mathrm{msign}$ |
| w/o outer un-whiten (bare msign) | `kl-diag-polar-flatout-lora` | $C_B=B^\top P B$, $Q$ (inner only) | $\mathrm{msign}$, not un-whitened |

**Averaged loss.** Replacing the per-sample bound $\max_i |\langle G_i,\Delta W\rangle|\le\tau$
with the batch RMS $\bigl(\tfrac1n\sum_i \langle G_i,\Delta W\rangle^2\bigr)^{1/2}\le\tau$
makes the constraint exactly $\|P^{1/2}\Delta W Q^{1/2}\|_F\le\tau$ under
$\Sigma\approx Q\otimes P$ — no outergradient set and no rank-one/leverage lemma
are needed, since those exist only to relax the max. Spectral becomes Frobenius,
so the LMO solution loses $\mathrm{msign}$:
$D_A = C_B^{-1}\widehat M_A Q^{-1}$, $D_B = P^{-1}\widehat M_B C_A^{-1}$.
The magnitude rule and the $p,q$ updates are unchanged.

**Product structure.** Applying the same per-sample LMO to each factor as its own
weight matrix, with per-factor fits $\Sigma_A\approx Q_A\otimes P_A$ and
$\Sigma_B\approx Q_B\otimes P_B$, gives
$D_A = P_A^{-1/2}\,\mathrm{msign}(P_A^{-1/2}\widehat M_A Q_A^{-1/2})\,Q_A^{-1/2}$
and symmetrically for $B$. The partner Gram $C_B=B^\top P B$ is replaced by $A$'s
own $r\times r$ gradient Gram, and $A$ and $B$ no longer share one layer metric
$(P,Q)$. The magnitude rule $\rho=\eta/(\|A\|_2+\|B\|_2)$ is kept, so only the
direction changes.

**Outer un-whitening.** Dropping the trailing $C_B^{-1/2}(\cdot)Q^{-1/2}$ leaves
$D_A=\mathrm{msign}(C_B^{-1/2}\widehat M_A Q^{-1/2})$. This is no longer the
solution of the LMO under $\|C_B^{1/2}\Delta A Q^{1/2}\|_2\le\tau$; it solves that
LMO under the *plain* $\|\Delta A\|_2\le\tau$ with a curvature-whitened gradient,
so the curvature picks the frame but the constraint leaves the curvature norm.
Because $\mathrm{msign}$ has every nonzero singular value equal to $1$,
$\|D_A\|_2=1$ by construction and the rescale $\Delta A=-\rho D_A/\|D_A\|_2$
collapses to $\Delta A=-\rho\,\mathrm{msign}(z)$: this arm needs no
$\sigma_{\max}$ estimate at all.

**Plotted:** final eval loss at 9000 steps vs learning rate (left) and the
best-lr trajectory (right), for the protagonist and the two arms, with AdamW as
the anchor; Δ vs the protagonist in σ-units.

**Why:** each arm deletes one premise of the derivation, so the position of each
curve's minimum and its best-lr trajectory separate what the per-sample
constraint contributes from what the product linearization contributes. The
discriminating feature is where each arm's best-lr final loss sits relative to
the two ends of the range — the protagonist and AdamW.

In [ ]:
P.derivation_ablation_panel(256)


### Figure 16. The $r\times r$ metric slot — Llama-3.2-1B openmath r256

Figure 15 varies the orthogonalization and the metric *power*. These four arms hold
both fixed and vary only what occupies the two $r\times r$ slots, $C_B=B^\top P B$ and
$C_A=A Q A^\top$, and whether the $d$-side diagonals are shared between the $A$- and
$B$-updates.

| arm | $r\times r$ slot | $d$-side diagonals |
|---|---|---|
| PoLoRA | $C_B=B^\top P B$ | one shared $(P,Q)$ |
| `rxr = I, shared P,Q` | $I$ | one shared $(P,Q)$ |
| `factorwise` | own-gradient Gram $P_A$ | separate $(P_A,Q_A)$, $(P_B,Q_B)$ |
| `factorwise + rxr = I` | $I$ | separate $(P_A,Q_A)$, $(P_B,Q_B)$ |

**Plotted:** final eval loss vs learning rate, and the best-lr trajectory, with
AdamW for scale and $\Delta$ against PoLoRA in $\sigma$-units.

**Why:** the first two rows differ only in the slot's contents; the first and third
differ in the contents *and* the sharing. `factorwise + rxr = I` is the fourth corner
of that 2x2, so the pair of comparisons separates the slot's contents from the
sharing instead of confounding them.

In [ ]:
P.rr_slot_panel(256)


## Figure 17. Per-rank final loss vs lr — Llama-3.2-1B openmath, all ranks (r16/r32/r64/r128/r256)

Two side-by-side panels, **Polar-LoRA (kl-diag)** and **AdamW**. For each, final eval loss
(9000 steps) vs lr, one curve per rank (color = rank, reversed viridis). Shared y-axis; x is
each optimizer's own lr grid. Final-step runs only (partial runs skipped). Exploratory all-ranks
view; the clean $r\ge32$ transfer figure (r16 excluded) lives in `paper_figs.ipynb`.

In [ ]:
# Final eval loss (9000 steps) vs lr, ALL ranks (Llama openmath). Two side-by-side panels:
# Polar-LoRA (kl-diag) | AdamW. Color = rank (reversed viridis: light = low rank, dark = high).
# Only runs that reached 9000 steps count (a partial run would plot an early-step loss against
# final losses). Exploratory all-ranks view; the clean r>=32 transfer figure lives in paper_figs.ipynb
# (r16's flat basin top is under-resolved). Shared y, windowed to the converged band (high-lr
# divergence runs off the top of the panel; clipped count printed).
import numpy as np
import matplotlib.cm as cm
from matplotlib.lines import Line2D
RANKS=[16,32,64,128,256]; HORIZON=9000
RANK_COLOR={r:c for r,c in zip(RANKS, cm.viridis_r(np.linspace(0.12,0.92,len(RANKS))))}
def _final_by_lr(where, rank):
    # Filters the cell_runs(_om(rank)) prefetch (cell 1) in memory instead of an own
    # load_runs(where=...) call -- same predicate, applied post-hoc; see the
    # cell_runs(_om(rank)) comment in cell 1 for why this is behavior-identical.
    pred = {**where, 'lora_r': rank, 'model_name': 'meta-llama/Llama-3.2-1B',
            'data_dir': (lambda d: 'openmath' in str(d))}
    out={}
    for cfg,evs in cell_runs(_om(rank)):
        if not _pred_matches(cfg, pred): continue
        ev=[(e['step'],e['eval_loss']) for e in (evs or []) if e.get('eval_loss') is not None]
        if not ev or ev[-1][0] < HORIZON: continue   # final losses only (skip partial runs)
        lr=float(cfg.get('lr')); out[lr]=min(out.get(lr,9e9), ev[-1][1])
    return dict(sorted(out.items()))
_PANELS=[('Polar-LoRA (kl-diag)',PROTO),('AdamW',ADAMW)]
panel_data=[]; allv=[]; empty=[]
for label,where in _PANELS:
    dd={rank:_final_by_lr(where,rank) for rank in RANKS}
    panel_data.append((label,dd))
    for rank,d in dd.items():
        if d: allv+=list(d.values())
        else: empty.append(f'{label} r{rank}')
# robust y-window: keep the converged band readable, let high-lr divergence run off-panel
med=float(np.median(allv)); conv=[v for v in allv if v < 3*med]
lo,hi=min(allv),max(conv); rng=hi-lo
ylo,yhi=lo-0.10*rng, hi+0.06*rng   # extra bottom pad so the lowest (r256) markers clear the axis
fig,axes=plt.subplots(1,len(_PANELS),figsize=(10.5,4.4),sharey=True)
for ax,(label,dd) in zip(axes,panel_data):
    for rank in RANKS:
        d=dd[rank]
        if d: ax.plot(list(d),list(d.values()),'o-',color=RANK_COLOR[rank],label=f'r={rank}')
    ax.set_xscale('log'); ax.set_xlabel('lr'); ax.set_title(label); ax.grid(alpha=.3)
    ax.set_ylim(ylo,yhi)
axes[0].set_ylabel('final eval loss (9000 steps)')
h_rank=[Line2D([],[],color=RANK_COLOR[r],marker='o',lw=2,label=f'r={r}') for r in RANKS]
axes[-1].legend(handles=h_rank,title='rank',loc='best',fontsize=8)
fig.suptitle('Final loss vs lr, all ranks (Llama openmath, 9000 steps)')
nclip=sum(v>yhi for v in allv)
if nclip: print(f'{nclip} high-lr point(s) above y-window (divergence), clipped at {yhi:.3f}')
if empty: print('NO 9000-step data (arm not run yet):', ', '.join(empty))
plt.tight_layout(); plt.show()

## Figure 18. Gauge diagnostics (exploratory) — openmath r256, per-pair over 112 LoRA pairs

Rougher/denser than `paper_figs.fig_gauge`: raw $\sigma_{max}(A)$, $\sigma_{max}(B)$, the per-pair ratio, BaLoRA residual, and factor norms, both arms. Reads the gauge diagnostic runs' logs directly; default palette. Populates as `gauge_kldiag_r256_bw` / `gauge_loramuon_r256_bw` run.

In [ ]:
# Exploratory gauge diagnostics: read the gauge runs' per-step optim_step events directly
# (per-pair min/median/max over the 112 LoRA pairs). Rougher than the polished paper_figs version.
import json, glob
import numpy as np
import matplotlib.pyplot as plt
def gtraj(group):
    for f in sorted(glob.glob(str(ROOT/'logs'/group/'run_info'/'logs'/'log_*.out'))):
        rows=[json.loads(l) for l in open(f) if '"balance_resid_median"' in l]
        rows=[r for r in rows if r.get('event')=='optim_step']
        if rows:
            g=lambda k: np.array([r.get(k,np.nan) for r in rows], float)
            return {k:g(k) for k in ('step','sigma_max_A_median','sigma_max_A_min','sigma_max_A_max',
                'sigma_max_B_median','sigma_max_B_min','sigma_max_B_max','sigma_ratio_median',
                'sigma_ratio_min','sigma_ratio_max','balance_resid_median','balance_resid_min',
                'balance_resid_max','norm_A_median','norm_B_median')}
    return None
ARMS={'ours (kl-diag)':'gauge_kldiag_r256_bw','LoRA-Muon step':'gauge_loramuon_r256_bw'}
D={k:gtraj(v) for k,v in ARMS.items()}
fig,ax=plt.subplots(2,2,figsize=(11,7))
for i,(lab,d) in enumerate(D.items()):       # per-pair sigma ratio
    if d is None: continue
    ax[0,0].fill_between(d['step'],d['sigma_ratio_min'],d['sigma_ratio_max'],color=f'C{i}',alpha=.15)
    ax[0,0].plot(d['step'],d['sigma_ratio_median'],f'C{i}',label=lab)
ax[0,0].axhline(1,color='k',ls=':',lw=.8); ax[0,0].set_yscale('log')
ax[0,0].set_title('per-pair sigma_max(A)/sigma_max(B)'); ax[0,0].set_xlabel('step'); ax[0,0].legend(fontsize=8)
d=D['ours (kl-diag)']                          # sigma_max A,B for ours (convergence)
if d is not None:
    ax[0,1].fill_between(d['step'],d['sigma_max_A_min'],d['sigma_max_A_max'],color='C0',alpha=.15)
    ax[0,1].plot(d['step'],d['sigma_max_A_median'],'C0',label='sigma_max(A)')
    ax[0,1].fill_between(d['step'],d['sigma_max_B_min'],d['sigma_max_B_max'],color='C3',alpha=.15)
    ax[0,1].plot(d['step'],d['sigma_max_B_median'],'C3',label='sigma_max(B)')
ax[0,1].set_title('ours: sigma_max(A),(B)  [band=112 pairs]'); ax[0,1].set_xlabel('step'); ax[0,1].legend(fontsize=8)
for i,(lab,d) in enumerate(D.items()):       # balance_resid
    if d is None: continue
    ax[1,0].fill_between(d['step'],d['balance_resid_min'],d['balance_resid_max'],color=f'C{i}',alpha=.13)
    ax[1,0].plot(d['step'],d['balance_resid_median'],f'C{i}',label=lab)
ax[1,0].axhline(0,color='k',ls=':',lw=.8); ax[1,0].set_ylim(0,1.05)
ax[1,0].set_title('BaLoRA balance_resid'); ax[1,0].set_xlabel('step'); ax[1,0].legend(fontsize=8)
for i,(lab,d) in enumerate(D.items()):       # factor norms
    if d is None: continue
    ax[1,1].plot(d['step'],d['norm_A_median'],f'C{i}',label=f'{lab} ||A||')
    ax[1,1].plot(d['step'],d['norm_B_median'],f'C{i}',ls='--',label=f'{lab} ||B||')
ax[1,1].set_title('factor Frobenius norms (median)'); ax[1,1].set_xlabel('step'); ax[1,1].legend(fontsize=7)
fig.suptitle('Gauge diagnostics (exploratory) - openmath r256, per-pair over 112 LoRA pairs')
plt.tight_layout(); plt.show()

## Figure 19. Does the op-norm self-balancing weaken at small rank? (r16 vs r256)

Protagonist (kl-diag) at matched lr=0.01, openmath. Op-norm ratio $\sigma_{max}(B)/\sigma_{max}(A)$ (ratio-of-medians; r16 predates the per-pair field) over training. r256 self-balances to ~1; r16 plateaus at ~0.88 (residual imbalance) — consistent with the mild r16 lr-shift (gauge self-corrects less at low rank).

In [ ]:
# r16 vs r256 op-norm self-balancing (protagonist, matched lr=0.01). ratio-of-medians
# sigma_max(B)/sigma_max(A) -> 1 = balanced. r16 uses the old-commit run (no per-pair field).
import json, glob
import numpy as np
import matplotlib.pyplot as plt
def otraj(group, lr_want=0.01):
    for f in sorted(glob.glob(str(ROOT/'logs'/group/'run_info'/'logs'/'log_*.out'))):
        lr=None; S=[]; rat=[]; br=[]
        for l in open(f):
            if '"event": "config"' in l:
                try: lr=float(json.loads(l).get('lr'))
                except: pass
            elif 'sigma_max_A_median' in l and '"event": "optim_step"' in l:
                e=json.loads(l); S.append(e['step'])
                rat.append(e['sigma_max_B_median']/e['sigma_max_A_median']); br.append(e['balance_resid_median'])
        if lr is not None and abs(lr-lr_want)<1e-9 and S:
            return np.array(S), np.array(rat), np.array(br)
    return None
ARMS=[('r16','e1_kldiag_llama32_openmath_r16r32_bw','C0'),
      ('r256','e1_kldiag_llama32_openmath_r256_bw','C3')]
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].axhline(1,color='k',ls='--',lw=.8)
for lab,grp,c in ARMS:
    d=otraj(grp)
    if d is None: continue
    ax[0].plot(d[0],d[1],c,lw=2,label=lab); ax[1].plot(d[0],d[2],c,lw=2,label=lab)
ax[0].set_xlabel('step'); ax[0].set_ylabel('sigma_max(B)/sigma_max(A) (ratio of medians)')
ax[0].set_title('op-norm self-balancing vs rank'); ax[0].legend()
ax[1].set_ylim(0,1.05); ax[1].set_xlabel('step'); ax[1].set_ylabel('balance_resid (median)')
ax[1].set_title('BaLoRA balance residual vs rank'); ax[1].legend()
fig.suptitle('op-norm self-balancing weakens at small rank (protagonist, openmath, lr=0.01)')
plt.tight_layout(); plt.show()

## Figure 20. Solved magnitude rule (cw_solved_rho) — live sweep tracking, Llama-3.2-1B openmath r256

| | |
|---|---|
| solved arm | `logs/solvedrho_kldiag_llama32_openmath_r256_bw` — `kl-diag-polar-lora` + `--cw_solved_rho`, lr ∈ {3e-3, 1e-2, 3e-2}, seed 0 |
| baseline arm | `logs/e1_kldiag_llama32_openmath_r256_bw` — same optimizer, bound $\rho=\eta/(\sigma_{max}(A)+\sigma_{max}(B))$, same lr grid |
| setting | Llama-3.2-1B, OpenMathInstruct-2 `packed_v1.1`, $r=256$, 9000 steps, eval every 250 |

Reads the two groups' `log_*.out` files directly (live-safe, tolerant of mid-write lines; re-run both cells to refresh — the loader/cache is not involved).

**Plotted:** held-out `eval_loss` vs step; one color per lr, solved arm solid, bound-ρ baseline dashed.

**Why:** the solved rule sizes the factor step by the root of $\rho t + \rho^2 = \eta$ with $t = \|B U_A + U_B A\|_2$ measured, keeping $\|\Delta(BA)\|_2 \le \eta$ while spending budget the bound rule leaves as triangle-inequality slack. Matched-lr solved curves at or below the dashed baseline would support the resize; divergence/NaN (most likely at lr 3e-2, where the realized product step grows most) would falsify the sizing being safe at this scale.

In [ ]:
# Live tracking (compute): eval trajectories straight from the groups' logs.
# Direct log read (same pattern as the gauge cells) — live-safe mid-run; the
# loader/logs-cache is bypassed on purpose so re-running refreshes.
import json, glob

SOLVED_GROUPS = ['solvedrho_kldiag_llama32_openmath_r256_bw',
                 'solvedrho_kldiag_llama32_openmath_r256_lrext_bw']  # lr ext {3e-4, 1e-3}
BOUND_GROUP = 'e1_kldiag_llama32_openmath_r256_bw'

def _eval_trajs(group):
    """{lr: (steps, losses)} from each log_*.out's config + eval events."""
    out = {}
    for f in sorted(glob.glob(str(ROOT/'logs'/group/'run_info'/'logs'/'log_*.out'))):
        lr = None; S = []; L = []
        for line in open(f):
            if '"event": "config"' in line:
                try: lr = float(json.loads(line).get('lr'))
                except Exception: pass
            elif '"event": "eval"' in line:
                try:
                    e = json.loads(line)
                    if e.get('eval_loss') is not None:
                        S.append(e['step']); L.append(e['eval_loss'])
                except Exception: pass  # mid-write partial line
        if lr is not None and S:
            out[lr] = (S, L)
    return dict(sorted(out.items()))

solved_trajs = {}
for g in SOLVED_GROUPS:
    solved_trajs.update(_eval_trajs(g))
solved_trajs = dict(sorted(solved_trajs.items()))
bound_trajs = _eval_trajs(BOUND_GROUP)
for lab, d in (('solved', solved_trajs), ('bound', bound_trajs)):
    print(lab, {lr: (S[-1], round(L[-1], 4)) for lr, (S, L) in d.items()})


In [ ]:
# Live tracking (plot): solved (solid) vs bound-rho baseline (dashed).
# One distinct color per plotted curve (arm + lr in the label) — no color reuse
# across arms, so skipping lrs on one side cannot produce a same-color pair.
import matplotlib.pyplot as plt
from lora_playground.plotting import distinct_palette

SKIP_SOLVED = {0.01, 0.03}  # off-target high-lr arms, excluded from the panel

curves = [(f'bound lr={lr:g}', '--', 1.5, S, L) for lr, (S, L) in bound_trajs.items()]
curves += [(f'solved lr={lr:g}', '-', 2.0, S, L)
           for lr, (S, L) in solved_trajs.items() if lr not in SKIP_SOLVED]
fig, ax = plt.subplots(figsize=(8, 5))
palette = distinct_palette(len(curves), reserved=[])
for color, (lab, ls, lw, S, L) in zip(palette, curves):
    ax.plot(S, L, ls=ls, lw=lw, color=color, label=lab)
ax.set_xlabel('step'); ax.set_ylabel('eval loss')
ax.set_title('kl-diag-polar-lora, solved vs bound magnitude — Llama-3.2-1B openmath r256')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.tight_layout(); plt.show()


## Live sweep tracking — in-flight E2 arms, Llama-3.2-1B openmath r256

Figures 20–22 share one helper, defined next.

In [ ]:
# Figures 21-23 below track in-flight sweeps. cell_runs() memoises per cell;
# call P.clear_runs_cache() once when a sweep has advanced and you want fresh
# numbers. (It used to force a re-read on EVERY call, which made these the
# slowest cells in the notebook.)


### Figure 21. Live tracking — protagonist $\beta_2$ sweep

Protagonist with $\beta_2\in\{0.81,0.909,0.956,0.979\}$ against the shipped $0.99$.
The $P,Q$ metric is accumulated *after* the step that consumes it
(`optim.py:2020-2024` applies, `:2068-2069` then folds in $g_t$), so it is stale by
one step — 1% of the $\approx100$-step EMA window at $\beta_2=0.99$, but $\approx19\%$
once the window shrinks to $1/(1-\beta_1^2)\approx5.3$ steps.

**Plotted:** eval loss vs step per $\beta_2$, and per-cell progress.
**Why:** whether the curves separate by $\beta_2$ at all. Read together with Figure 21.

In [ ]:
P.beta2_panel(256)


### Figure 22. Live tracking — AdamW $\beta_2$ control

AdamW on the same $\beta_2$ grid at its tuned $\eta=10^{-4}$. All four pre-existing
AdamW runs at this setting use $\beta_2=0.999$, so there was no coverage before.

**Plotted:** eval loss vs step per $\beta_2$, and per-cell progress.
**Why:** the control for Figure 20. If $\beta_2$ moves neither optimizer, the
workload's gradient shapes are simply stable; if it moves AdamW but not the
protagonist, the protagonist's max-normalized diagonal metric is insensitive to
the window.

In [ ]:
P.adamw_beta2_panel(256)


### Figure 23. Magnitude rule — naive $\rho=\eta$ vs the PoLoRA rule

$\rho=\eta$ flat instead of $\rho=\eta/(\|A\|_2+\|B\|_2)$ (`optim.py:1960-1961`), so each
factor update is normalised to spectral norm $\eta$. The $\sigma_{\max}$ rescale is
kept, so this swaps the magnitude *rule* rather than removing magnitude control.

**Plotted:** eval loss vs step per learning rate, and per-cell progress.
**Why:** $\eta$ means something different here — the protagonist divides by
$\|A\|_2+\|B\|_2$, which is $O(1\text{–}10)$ and grows — so the grid is shifted down
and the swept axis is the learning rate rather than $\beta_2$.

In [ ]:
P.magnitude_rule_panel(256)
